In [11]:
import numpy as np
from scipy.integrate import fixed_quad, quad
import os
import plotly.graph_objects as go
from scipy.special import j0, jv
import pandas as pd

In [12]:
# Leitura do arquivo com separação por espaços
data_atlas = pd.read_csv(
    "../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70  # lê apenas as 70 primeiras linhas
)

x_atlas = data_atlas[0].to_numpy()
y_atlas = data_atlas[1].to_numpy()
y_error_atlas = data_atlas[2].to_numpy()

/tmp/ipykernel_9988/3003301750.py:2: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [13]:
# === Global Configuration and Constants ===
start_sqrt_s = 101  # Global parameter controlling energy scale
b_0 = (33 - 6) / (12 * np.pi)  # β0 for nf=3
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0

lst_sigma_tot_born = []
lst_sqrt_s = []
lst_error = []

s0 = 1.0  # GeV^2

epsilon_atlas = 0.0732

max_sqrt_s = 13000
step = 100
n_points = 10000

model_params = {
    'atlas': {
        'pl':  {'mg': 0.417, 'a1': 1.563, 'a2': 2.22}
    }
}

epsilon_values = {
    'atlas': epsilon_atlas
}

lst_amp_born = []
lst_sqrt_s = []
lst_sigma_tot_born = []




In [14]:
# === Auxiliary Functions for Physical Model ===
def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)

def get_m2_function(mass_model):
    return m2_pl

def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, phi, mg, a1, a2, m2_func, q):
    q2 = q ** 2
    qk_cos = q * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)


# -------------------------------
# Inner integral (over phi)
# -------------------------------
def phi_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    integrand = lambda phi: k * (T_1(k, phi, mg, a1, a2, m2_func, q) - 
                                 T_2(k, phi, mg, a1, a2, m2_func, q))
    result, _ = fixed_quad(integrand, 0, 2*np.pi, n=n_points)
    return result

# -------------------------------
# Outer integral (over k)
# -------------------------------
def k_integral(k, mg, a1, a2, m2_func, q, n_points=10000):
    return phi_integral(k, mg, a1, a2, m2_func, q, n_points)

# -------------------------------
# Double integral computation
# -------------------------------
def compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points=10000):
    result, _ = fixed_quad(
        lambda k: k_integral(k, mg, a1, a2, m2_func, q, n_points),
        0, sqrt_s_val, 
        n=n_points
    )
    return result

def born_amp(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + 0.25 * t

    first_term = s**(alpha_pomeron)
    second_term = 1/(1**(alpha_pomeron-1))
    
    regge_factor = first_term * second_term
    return 1j * 8.0 * regge_factor * diff_T

def sigma_tot_born(amp_born_value, s):
    return amp_born_value.imag / s * 0.389379323


In [15]:

def calculate_born_cross_sections(start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points=10000):
    """Calculate cross sections for all sqrt_s values"""
    
    # Generate array of sqrt_s values
    sqrt_s_values = np.arange(start_sqrt_s, max_sqrt_s + step, step)
    
    # Process each sqrt_s value
    lst_sigma_tot_born = []
    lst_sqrt_s = []
    lst_amp_born = []
    
    for sqrt_s_val in sqrt_s_values:
        # Compute the double integral
        q = 0  # Assuming q=0 as in the original code
        diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q, n_points)
        
        # Calculate amplitude and cross section
        s = sqrt_s_val * sqrt_s_val
        amp_born_value = born_amp(diff_T, s, epsilon, 0)
        sigma_tot_born_value = sigma_tot_born(amp_born_value, s)
        
        # Store results
        lst_sigma_tot_born.append(sigma_tot_born_value)
        lst_sqrt_s.append(sqrt_s_val)
        lst_amp_born.append(amp_born_value)
    
    return lst_sigma_tot_born, lst_sqrt_s, lst_amp_born



In [16]:
mass_model = 'pl'
ensemble = 'atlas'
m2_func = get_m2_function(mass_model)
params = model_params[ensemble][mass_model]
mg, a1, a2 = params['mg'], params['a1'], params['a2']
epsilon = epsilon_values[ensemble]

# Calculate all cross sections
lst_sigma_tot_born, lst_sqrt_s, lst_amp_born = calculate_born_cross_sections(
    start_sqrt_s, max_sqrt_s, step, mg, a1, a2, m2_func, epsilon, n_points
)

lst_s = [val**2 for val in lst_sqrt_s]

In [ ]:
n = 10000
q_max = 0.1


# -------------------------------
# Full chi(s,b) including k, phi, and new q integral
# -------------------------------
def chi_integral(sqrt_s_values, mg, a1, a2, m2_func, epsilon, b):
    """
    Computes chi(s,b) = (1/s) ∫_0^qmax q dq J0(b q) [i 8 s^(1+ε) (∫_0^√s k dk ∫_0^2π dφ (T1-T2)) ]
    """
    chi_list = []

    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over q
        def q_integral(q):
            t = -q**2
            # compute existing double integral over k and phi
            diff_T = compute_double_integral(sqrt_s_val, mg, a1, a2, m2_func, q)
            return (q * j0(b * q) * born_amp(diff_T, s, epsilon, t))/s

        # integrate real and imaginary parts separately
        real_part, _ = quad(lambda q: q_integral(q).real, 0, q_max, limit=n, epsabs=1e-10, epsrel=1e-10)
        imag_part, _ = quad(lambda q: q_integral(q).imag, 0, q_max, limit=n, epsabs=1e-10, epsrel=1e-10)

        chi_list.append((real_part + 1j * imag_part))

    return chi_list


In [ ]:
amp_list = []
b_max = 30

# -------------------------------
# Eikonal amplitude A_eik(s,t)
# -------------------------------
def eikonal_amplitude(sqrt_s_values, mg, a1, a2, m2_func, epsilon):


    for sqrt_s_val in sqrt_s_values:
        s = sqrt_s_val**2

        # integrand over b
        def b_integrand(b):
            chi_val = chi_integral([sqrt_s_val], mg, a1, a2, m2_func, epsilon, b)[0]
            return b * (1 - np.exp(1j * chi_val))

        # integrate real and imaginary parts separately
        real_part, _ = quad(lambda b: b_integrand(b).real, 0, b_max, limit=n, epsabs=1e-10, epsrel=1e-10)
        imag_part, _ = quad(lambda b: b_integrand(b).imag, 0, b_max, limit=n, epsabs=1e-10, epsrel=1e-10)

        A_eik = 1j * s * (real_part + 1j * imag_part)
        amp_list.append(A_eik)

    return amp_list


# Example usage
A_eik_values = eikonal_amplitude(lst_sqrt_s, mg, a1, a2, m2_func, epsilon)
print(A_eik_values)


[1340449.8660491698j, 5752217.6092584245j, 13510153.415051999j, 24769367.856178135j, 39641052.7786956j, 58212453.85013442j, 80555580.51877886j, 106731845.34451506j, 136794843.9332918j, 170792157.8767316j, 208766593.0112588j, 250757067.73584414j, 296799272.5494823j, 346926173.53300726j, 401168405.6365328j, 459554585.88825274j, 522111566.9783491j, 588864645.5147203j, 659837735.192106j, 735053512.3702947j, 814533539.6514567j, 898298371.6949193j, 986367646.5314401j, 1078760164.9230394j, 1175493959.7719245j, 1276586357.1876416j, 1382054030.496211j, 1491913048.2501645j, 1606178917.0904787j, 1724866620.1856887j, 1847990651.823983j, 1975565048.672067j, 2107603418.1065302j, 2244118963.9756155j, 2385124510.1002603j, 2530632521.7642326j, 2680655125.420454j, 2835204126.808218j, 2994291027.6468883j, 3157927041.0507393j, 3326123105.7922587j, 3498889899.5296783j, 3676237851.0950966j, 3858177151.930927j, 4044717766.747059j, 4235869443.4759746j, 4431641722.581027j, 4632043945.778381j, 4837085264.212364

In [19]:
def sigma_tot_eik(amp, s):
    return (4*np.pi)/s * amp.imag * 0.389379323   

lst_sigma_tot_eik = [sigma_tot_eik(amp, s) for amp, s in zip(A_eik_values, lst_s)]
print(lst_sigma_tot_eik)


[642.9698044468083, 696.668127622051, 729.6423597301372, 753.7180701819616, 772.7739573679452, 788.5871797162829, 802.1255415196279, 813.9762545950455, 824.5232302477812, 834.0316284482393, 842.6924744416785, 850.648063395525, 858.0073115962437, 864.8554871423405, 871.260625677095, 877.2779024513067, 882.9526966019454, 888.3227914715242, 893.4199882330232, 898.2713114017747, 902.8999243725237, 907.3258350026373, 911.5664465968091, 915.6369933160646, 919.5508879848902, 923.3200026747772, 926.9548971005188, 930.4650060878313, 933.85879462014, 937.1438869886082, 940.32717506956, 943.4149096671098, 946.4127780032222, 949.3259698083696, 952.1592339741622, 954.9169273399561, 957.6030568927704, 960.2213164223219, 962.7751184841906, 965.2676223748472, 967.7017587022397, 970.0802510393254, 972.4056350666517, 974.6802755462936, 976.9063814148182, 979.0860192424943, 981.2211252654977, 983.3135161704987, 985.3648987820885, 987.3768787860291, 989.3509686011248, 991.2885944977393, 993.1911030483553,

In [20]:
def add_iterative_curve(fig, x_data, y_data, 
                        curve_name:str=None, color:str='blue', line_type:str='lines+markers'):

    fig.add_trace(go.Scatter(
    x = x_data,
    y = y_data,
    mode=line_type, 
    name=curve_name,
    line=dict(
        color=color,
        width=2),
    marker=dict(size=4))
)
    fig.update_xaxes(gridcolor='lightgray')
    fig.update_yaxes(gridcolor='lightgray')


fig_sigma = go.Figure()

add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_born, curve_name='sigma tot born')
add_iterative_curve(fig_sigma, lst_sqrt_s, lst_sigma_tot_eik, curve_name='sigma tot eikonal', color='red')


# Add ATLAS data
fig_sigma.add_trace(go.Scatter(
    x=x_atlas,
    y=y_atlas,
    mode='markers',
    marker=dict(
        color='black',
        size=6,
        symbol='square'
    ),
    error_y=dict(
        type='data',
        array=y_error_atlas,
        visible=True
    ),
    name='ATLAS Data'
))

# Configure layout
fig_sigma.update_layout(
    title='Sigma Tot vs. sqrt(s) - b = [0, 10], q = [0, 0.1]',
    xaxis=dict(
        title='sqrt(s) [GeV]',
        type='log',
    ),
    yaxis=dict(
        title='Sigma Tot [mb]',
    ),
    showlegend=True,
    legend=dict(
        title='Model/Data'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_sigma.update_xaxes(gridcolor='lightgray')
fig_sigma.update_yaxes(gridcolor='lightgray')

fig_sigma.show(renderer = 'browser')

Opening in existing browser session.
